In [1]:
# =============================================================================
# CELL 1 — Mount Google Drive & set paths
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import pandas as pd
import os

# --- Paths -------------------------------------------------------------------
DATA_DIR: Path = Path("/content/drive/MyDrive/Sleep")
GLOBAL_META_PATH = DATA_DIR / "_global_meta.npz"
MANIFEST_PATH    = DATA_DIR / "cassette_manifest.csv"
SUBJ_SHARD_PATH  = DATA_DIR / "subject_to_shard.csv"

print("DATA_DIR exists :", DATA_DIR.exists())
print("\nFiles in Sleep/ :")
for p in sorted(DATA_DIR.iterdir()):
    size_mb = p.stat().st_size / 1e6
    print(f"  {p.name:35s}  {size_mb:10.1f} MB")

Mounted at /content/drive
DATA_DIR exists : True

Files in Sleep/ :
  _global_meta.npz                            0.0 MB
  cassette_manifest.csv                       0.0 MB
  checkpoints                                 0.0 MB
  checkpoints_01_mcaf_eeg_5fold               0.0 MB
  checkpoints_02_liteeeg_fusion               0.0 MB
  checkpoints_02_liteeeg_fusion_single         0.0 MB
  checkpoints_MSDCA                           0.0 MB
  checkpoints_MSDCA_70_15_15                  0.0 MB
  checkpoints_cv                              0.0 MB
  checkpoints_tap_mcafnet                     0.0 MB
  checkpoints_tap_mcafnet_single              0.0 MB
  shard_01.npz                              927.3 MB
  shard_02.npz                              918.8 MB
  shard_03.npz                              946.6 MB
  shard_04.npz                              871.2 MB
  shard_05.npz                              924.3 MB
  shard_06.npz                              893.4 MB
  shard_07.npz                

In [2]:
# =============================================================================
# CELL 2 — Discover all shard files
# =============================================================================

shard_paths = sorted(DATA_DIR.glob("shard_*.npz"))
print(f"Found {len(shard_paths)} shard files:")
for p in shard_paths:
    print(f"  {p.name}  ({p.stat().st_size / 1e9:.2f} GB)")

assert len(shard_paths) > 0, "No shard files found. Check the path."

Found 10 shard files:
  shard_01.npz  (0.93 GB)
  shard_02.npz  (0.92 GB)
  shard_03.npz  (0.95 GB)
  shard_04.npz  (0.87 GB)
  shard_05.npz  (0.92 GB)
  shard_06.npz  (0.89 GB)
  shard_07.npz  (0.94 GB)
  shard_08.npz  (0.87 GB)
  shard_09.npz  (0.86 GB)
  shard_10.npz  (0.70 GB)


In [3]:
# =============================================================================
# CELL 3 — Read and print global metadata
# =============================================================================

with np.load(GLOBAL_META_PATH, allow_pickle=False) as d:
    global_meta = {k: d[k] for k in d.files}

print("=" * 60)
print("GLOBAL METADATA — _global_meta.npz")
print("=" * 60)
for k, v in global_meta.items():
    if v.ndim == 0:
        print(f"  {k:20s}: {v.item()}")
    else:
        print(f"  {k:20s}: shape={v.shape}  dtype={v.dtype}")
        print(f"    {v.tolist()}")

# ---- Channel order is the key info we need ---------------------------------
channel_names = [str(x) for x in global_meta["channel_names"]]
print("\n" + "=" * 60)
print("X CHANNEL ORDER")
print("=" * 60)
for i, name in enumerate(channel_names):
    print(f"  X[:, {i}, :]  ->  {name}")

label_names = [str(x) for x in global_meta["label_names"]]
print("\nLabel encoding (y):")
for i, name in enumerate(label_names):
    print(f"  y == {i}  ->  {name}")

sfreq          = int(global_meta["sfreq"])
epoch_duration = int(global_meta["epoch_duration"])
print(f"\nSampling rate   : {sfreq} Hz")
print(f"Epoch duration  : {epoch_duration} s")
print(f"Samples / epoch : {sfreq * epoch_duration}")

GLOBAL METADATA — _global_meta.npz
  channel_names       : shape=(5,)  dtype=<U14
    ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'EMG submental', 'subject_id']
  label_names         : shape=(5,)  dtype=<U3
    ['W', 'N1', 'N2', 'N3', 'REM']
  sfreq               : 100
  epoch_duration      : 30

X CHANNEL ORDER
  X[:, 0, :]  ->  EEG Fpz-Cz
  X[:, 1, :]  ->  EEG Pz-Oz
  X[:, 2, :]  ->  EOG horizontal
  X[:, 3, :]  ->  EMG submental
  X[:, 4, :]  ->  subject_id

Label encoding (y):
  y == 0  ->  W
  y == 1  ->  N1
  y == 2  ->  N2
  y == 3  ->  N3
  y == 4  ->  REM

Sampling rate   : 100 Hz
Epoch duration  : 30 s
Samples / epoch : 3000


In [4]:
# =============================================================================
# CELL 4 — Read manifest and subject-shard mapping
# =============================================================================

manifest = pd.read_csv(MANIFEST_PATH)
subj_shard = pd.read_csv(SUBJ_SHARD_PATH)

print("=" * 60)
print("MANIFEST (cassette_manifest.csv)")
print("=" * 60)
print("Shape:", manifest.shape)
print("Columns:", manifest.columns.tolist())
print()
print(manifest.head(10).to_string(index=False))
print()
print("Shard distribution in manifest:")
print(manifest.groupby("shard_id").agg(
    n_recordings=("recording_key", "count"),
    n_subjects=("subject", "nunique"),
    n_epochs_kept=("n_epochs_kept", "sum"),
).to_string())

print("\n" + "=" * 60)
print("SUBJECT → SHARD MAPPING")
print("=" * 60)
print("Shape:", subj_shard.shape)
print(subj_shard.head(10).to_string(index=False))
print()
print("Subjects per shard:")
print(subj_shard.groupby("shard_id")["subject"].count().to_string())

MANIFEST (cassette_manifest.csv)
Shape: (153, 10)
Columns: ['recording_key', 'shard_id', 'subject', 'night', 'n_epochs_total', 'n_epochs_kept', 'n_w_total', 'n_w_kept', 'n_sleep_kept', 'missing_channels']

recording_key  shard_id  subject  night  n_epochs_total  n_epochs_kept  n_w_total  n_w_kept  n_sleep_kept  missing_channels
     SC4001E0         0        0      1            2650            773       1997       120           653               NaN
     SC4002E0         0        0      2            2830           1064       1885       120           944               NaN
     SC4251E0         0       25      1            2760            959       1921       120           839               NaN
     SC4252E0         0       25      2            2666            930       1855       120           810               NaN
     SC4281G0         0       28      1            2788           1054       1854       120           934               NaN
     SC4282G0         0       28      2           

In [5]:
# =============================================================================
# CELL 5 — Inspect one shard in detail
# =============================================================================

sample_shard_path = shard_paths[0]
print(f"Inspecting: {sample_shard_path.name}\n")

with np.load(sample_shard_path, allow_pickle=False) as d:
    print("Keys inside shard:", d.files)
    for k in d.files:
        arr = d[k]
        print(f"  {k:8s}: shape={arr.shape}  dtype={arr.dtype}")

    X_sample = d["X"]
    y_sample = d["y"]

print("\nX sample stats (per channel):")
for i, name in enumerate(channel_names):
    ch = X_sample[:, i, :]
    print(f"  ch {i} ({name:15s}): "
          f"min={ch.min():12.4f}  max={ch.max():12.4f}  "
          f"mean={ch.mean():10.4f}  std={ch.std():10.4f}  "
          f"nan={np.isnan(ch).sum()}")

print("\ny distribution:")
for i, name in enumerate(label_names):
    n = int((y_sample == i).sum())
    pct = 100 * n / len(y_sample)
    print(f"  {name:>4}: {n:>8,}  ({pct:5.2f}%)")

# Check subject_id is constant per row
subj_in_X = X_sample[:, channel_names.index("subject_id"), 0]
print(f"\nsubject_id in X (first 10 epochs): {subj_in_X[:10]}")
print(f"subject_id unique in this shard   : {np.unique(subj_in_X)}")

Inspecting: shard_01.npz

Keys inside shard: ['X', 'y']
  X       : shape=(15454, 5, 3000)  dtype=float32
  y       : shape=(15454,)  dtype=int8

X sample stats (per channel):
  ch 0 (EEG Fpz-Cz     ): min=     -0.0002  max=      0.0002  mean=   -0.0000  std=    0.0000  nan=0
  ch 1 (EEG Pz-Oz      ): min=     -0.0002  max=      0.0002  mean=    0.0000  std=    0.0000  nan=0
  ch 2 (EOG horizontal ): min=     -0.0006  max=      0.0006  mean=    0.0000  std=    0.0000  nan=0
  ch 3 (EMG submental  ): min=     -0.0000  max=      0.0000  mean=    0.0000  std=    0.0000  nan=0
  ch 4 (subject_id     ): min=      0.0000  max=     61.0000  mean=   37.6662  std=   18.9764  nan=0

y distribution:
     W:    1,920  (12.42%)
    N1:    2,069  (13.39%)
    N2:    6,684  (43.25%)
    N3:    1,906  (12.33%)
   REM:    2,875  (18.60%)

subject_id in X (first 10 epochs): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
subject_id unique in this shard   : [ 0. 25. 28. 33. 41. 51. 60. 61.]


In [6]:

# =============================================================================
# CELL 6 — Standard extraction: X_signals, y, subject_id
# =============================================================================

# Symbolic channel indices (from global_meta)
SIGNAL_CHANNEL_INDICES = [i for i, n in enumerate(channel_names)
                          if n != "subject_id"]
SUBJECT_ID_INDEX       = channel_names.index("subject_id") \
                          if "subject_id" in channel_names else None

print("Signal channel indices in X:", SIGNAL_CHANNEL_INDICES)
print("Subject_id index in X     :", SUBJECT_ID_INDEX)


def load_shard_xy(shard_path: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Load a shard and return:
      X_signals : (n, 4, 3000) float32
      y         : (n,)         int8
      subject   : (n,)         int16
    """
    with np.load(shard_path, allow_pickle=False) as d:
        X_full = d["X"]
        y      = d["y"]

    X_signals = X_full[:, SIGNAL_CHANNEL_INDICES, :].astype(np.float32, copy=False)

    if SUBJECT_ID_INDEX is not None:
        subject = X_full[:, SUBJECT_ID_INDEX, 0].astype(np.int16)
    else:
        subject = np.zeros(len(y), dtype=np.int16)

    return X_signals, y, subject


# ---- Test on the first shard ------------------------------------------------
X_sample, y_sample, subj_sample = load_shard_xy(shard_paths[0])

print("\n" + "=" * 60)
print("EXTRACTED SHAPES")
print("=" * 60)
print(f"X_signals : {X_sample.shape}   dtype={X_sample.dtype}")
print(f"y         : {y_sample.shape}   dtype={y_sample.dtype}")
print(f"subject   : {subj_sample.shape}   dtype={subj_sample.dtype}")
print(f"\nX_signals per-epoch tensor (n, n_channels, n_samples)")
print(f"  n_channels = {X_sample.shape[1]}  -> {[channel_names[i] for i in SIGNAL_CHANNEL_INDICES]}")
print(f"  n_samples  = {X_sample.shape[2]}  -> {sfreq} Hz x {epoch_duration} s")

Signal channel indices in X: [0, 1, 2, 3]
Subject_id index in X     : 4

EXTRACTED SHAPES
X_signals : (15454, 4, 3000)   dtype=float32
y         : (15454,)   dtype=int8
subject   : (15454,)   dtype=int16

X_signals per-epoch tensor (n, n_channels, n_samples)
  n_channels = 4  -> ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'EMG submental']
  n_samples  = 3000  -> 100 Hz x 30 s


## Third Phase: 5-Fold Cross-Validation

In [7]:
import os, json, math, random, gc, time
from pathlib import Path
from typing import Iterator, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler

from google.colab import drive
drive.mount('/content/drive')

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
DATA_DIR   = Path("/content/drive/MyDrive/Sleep")
SPLIT_PATH = DATA_DIR / "splits.json"
# Base checkpoint directory for all cross-validation folds
CKPT_BASE_DIR   = DATA_DIR / "checkpoints_cv"
CKPT_BASE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Reproducibility
# -----------------------------------------------------------------------------
SEED = 42
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)

# -----------------------------------------------------------------------------
# Signal / dataset constants
# -----------------------------------------------------------------------------
SFREQ           = 100                # Hz
EPOCH_SEC       = 30
SAMPLES_PER_EPOCH = SFREQ * EPOCH_SEC  # 3000
N_CLASSES       = 5
CLASS_NAMES     = ["W", "N1", "N2", "N3", "REM"]

# Channels kept in X — order must be [Fpz-Cz, Pz-Oz, EOG]
KEEP_CHANNELS   = ["EEG Fpz-Cz", "EEG Pz-Oz", "EOG horizontal"]
N_CHANNELS      = len(KEEP_CHANNELS)

# -----------------------------------------------------------------------------
# STFT parameters (from paper §2.5.2)
# -----------------------------------------------------------------------------
STFT_NPERSEG    = 256         # 256-point STFT
STFT_NOVERLAP   = 128         # 50 % overlap of 2 s window (256 = 2 s × 100 Hz → hop = 128)
STFT_HOP        = STFT_NPERSEG - STFT_NOVERLAP  # 128 samples
STFT_FREQ_BINS  = STFT_NPERSEG // 2 + 1         # 129 → keep 128 (drop Nyquist)
STFT_TIME_BINS  = 29          # (3000 - 256) / 128 + 1 = 22.4 → 29?
# We'll compute STFT_TIME_BINS dynamically from the actual signal.

# -----------------------------------------------------------------------------
# Model hyper-parameters (from paper §2.5.2)
# -----------------------------------------------------------------------------
FEATURE_DIM     = 128         # D
N_ATTN_HEADS    = 4           # H
N_ATTN_LAYERS   = 2           # NA
CONV_KERNEL     = 3           # NC
CONV_GROUPS     = 3           # depth-wise separable groups
DROPout         = 0.5         # dropout rate
FC_HIDDEN       = 512         # HF

# -----------------------------------------------------------------------------
# Training hyper-parameters
# -----------------------------------------------------------------------------
BATCH_SIZE      = 128
NUM_EPOCHS      = 100
LR              = 1e-3
WEIGHT_DECAY    = 1e-2
WARMUP_EPOCHS   = 5
LABEL_SMOOTHING = 0.1
GRAD_CLIP       = 1.0
NUM_WORKERS     = 0           # Colab: keep 0 to avoid pickling issues
PIN_MEMORY      = True
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")
print(f"Feature dim D={FEATURE_DIM}, heads H={N_ATTN_HEADS}, layers NA={N_ATTN_LAYERS}")
print(f"Batch size: {BATCH_SIZE}, LR: {LR}, Epochs: {NUM_EPOCHS}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Feature dim D=128, heads H=4, layers NA=2
Batch size: 128, LR: 0.001, Epochs: 100


In [8]:
# =============================================================================
# CELL 2 — Generate 5-fold subject-wise splits
# =============================================================================

from sklearn.model_selection import GroupKFold, GroupShuffleSplit

# ---- Load manifest to know which subjects exist -----------------------------
manifest = pd.read_csv(DATA_DIR / "cassette_manifest.csv")
all_subjects = sorted(manifest["subject"].unique().tolist())
n_subjects = len(all_subjects)
print(f"Total subjects: {n_subjects}")

# ---- Create 5-fold cross-validation splits ----------------------------------
N_FOLDS = 5
# GroupKFold ensures subjects are not split across folds
gkf = GroupKFold(n_splits=N_FOLDS)

# We'll use the subject IDs themselves as the 'groups' for GroupKFold
# The 'X' and 'y' arguments here are just placeholders for the actual data
# we care about the split of 'groups' (subjects).
subject_indices = np.arange(n_subjects) # Indices corresponding to all_subjects

# Store the subject splits for each fold
# Each element will be (train_val_subjects_list, test_subjects_list)
subject_cv_splits = []

print(f"Generating {N_FOLDS}-fold cross-validation splits...")
for fold_idx, (train_val_idx, test_idx) in enumerate(gkf.split(
    subject_indices,  # X: placeholder
    subject_indices,  # y: placeholder
    groups=all_subjects # groups: the actual subjects to split
)):
    fold_all_subjects_arr = np.array(all_subjects)
    train_val_subjects_fold = fold_all_subjects_arr[train_val_idx].tolist()
    test_subjects_fold      = fold_all_subjects_arr[test_idx].tolist()

    # Further split train_val into train and validation for early stopping
    # We'll use a fixed percentage for validation within each fold's training
    val_split_ratio = 0.15 / (1.0 - (1.0 / N_FOLDS)) # Roughly 15% of the 80% train_val
    gss = GroupShuffleSplit(n_splits=1, test_size=val_split_ratio, random_state=SEED + fold_idx)

    # The split method returns indices, so we need to map them back to subjects
    train_idx_inner, val_idx_inner = next(gss.split(
        np.arange(len(train_val_subjects_fold)), # X: placeholder for inner split
        groups=train_val_subjects_fold            # groups: subjects in current train_val set
    ))

    train_subjects_fold = np.array(train_val_subjects_fold)[train_idx_inner].tolist()
    val_subjects_fold   = np.array(train_val_subjects_fold)[val_idx_inner].tolist()

    subject_cv_splits.append({
        "fold": fold_idx,
        "train_subjects": sorted(train_subjects_fold),
        "validation_subjects": sorted(val_subjects_fold),
        "test_subjects": sorted(test_subjects_fold),
    })

    print(f"  Fold {fold_idx}: Train={len(train_subjects_fold)}, Val={len(val_subjects_fold)}, Test={len(test_subjects_fold)}")

# Save all splits to a single JSON file
with open(SPLIT_PATH, "w") as f:
    json.dump(subject_cv_splits, f, indent=2)
print(f"✓ Saved {N_FOLDS}-fold subject splits to {SPLIT_PATH}")

# The rest of this cell's original content for class distribution checks
# will be moved or adapted within the CV loop as it needs fold-specific subjects.

Total subjects: 78
Generating 5-fold cross-validation splits...
  Fold 0: Train=50, Val=12, Test=16
  Fold 1: Train=50, Val=12, Test=16
  Fold 2: Train=50, Val=12, Test=16
  Fold 3: Train=51, Val=12, Test=15
  Fold 4: Train=51, Val=12, Test=15
✓ Saved 5-fold subject splits to /content/drive/MyDrive/Sleep/splits.json


In [9]:
# =============================================================================
# CELL 3 — Compute per-subject class counts (streaming, memory-safe)
# =============================================================================

SUBJECT_IDX_IN_X = 4  # index of subject_id in the raw shard X

# Load all subject & y arrays from shards (only those two, memory-safe)
subject_all_list, y_all_list = [], []
for sp in sorted(DATA_DIR.glob("shard_*.npz")):
    with np.load(sp, allow_pickle=False) as d:
        y = d["y"].astype(np.int8)
        subj = d["X"][:, SUBJECT_IDX_IN_X, 0].astype(np.int16)
    subject_all_list.append(subj)
    y_all_list.append(y)

subject_all = np.concatenate(subject_all_list)
y_all       = np.concatenate(y_all_list)
del subject_all_list, y_all_list
print(f"Total epochs loaded: {len(y_all):,}")

# Build per-subject counts
subjects_sorted = sorted(np.unique(subject_all).tolist())
counts = np.zeros((len(subjects_sorted), N_CLASSES), dtype=int)
for i, s in enumerate(subjects_sorted):
    counts[i] = np.bincount(
        y_all[subject_all == s], minlength=N_CLASSES
    )

df_counts = pd.DataFrame(counts, index=subjects_sorted, columns=CLASS_NAMES)
df_counts.index.name = "subject"
df_counts["TOTAL"] = df_counts[CLASS_NAMES].sum(axis=1)
print(df_counts.to_string())

# `class_dist_for_subjects` will be used within the CV loop.
def class_dist_for_subjects(subj_set):
    mask = np.isin(subject_all, list(subj_set))
    return np.bincount(y_all[mask], minlength=N_CLASSES)


Total epochs loaded: 147,596
           W   N1    N2   N3  REM  TOTAL
subject                                 
0        240  117   623  517  340   1837
1        240  201  1222  201  346   2210
2        240  278   947  214  342   2021
3        240  106   885  188  408   1827
4        240  303  1134  147  466   2290
5        240  158   833  249  248   1728
6        240  146   824  265  289   1764
7        240  173   795  384  366   1958
8        240  107   591  632  391   1961
9        240  100  1073  277  497   2187
10       240  182  1278   31  406   2137
11       240   31   898  240  309   1718
12       240  169   750  187  457   1803
13       120   57   497  147  172    993
14       240   56   790  296  446   1828
15       240   88   792  355  500   1975
16       240   97   907  293  455   1992
17       240   65  1015  403  427   2150
18       240  180   678  507  234   1839
19       240  190  1267  170  618   2485
20       240  159  1018    4  384   1805
21       240  193   929   92

## Fourth Phase: 5-Fold Cross-Validation Loop

In [14]:
# =============================================================================
# CELL 4 — 5-Fold Cross-Validation Loop
# =============================================================================

# Re-import necessary modules in case kernel restarted
import os, json, math, random, gc, time
from pathlib import Path
from typing import Iterator, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, f1_score, confusion_matrix,
    classification_report
)
from copy import deepcopy
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------------------------------------------------------
# Signal / dataset constants (moved from CELL 6)
# -----------------------------------------------------------------------------
SFREQ           = 100                # Hz
EPOCH_SEC       = 30
SAMPLES_PER_EPOCH = SFREQ * EPOCH_SEC  # 3000
N_CLASSES       = 5

# Channels kept in X — order must be [Fpz-Cz, Pz-Oz, EOG]
KEEP_CHANNELS   = ["EEG Fpz-Cz", "EEG Pz-Oz", "EOG horizontal"]
N_CHANNELS      = len(KEEP_CHANNELS)

# -----------------------------------------------------------------------------
# STFT parameters (from paper §2.5.2) (moved from CELL 6)
# -----------------------------------------------------------------------------
STFT_NPERSEG    = 256         # 256-point STFT
STFT_NOVERLAP   = 128         # 50 % overlap of 2 s window (256 = 2 s × 100 Hz → hop = 128)
STFT_HOP        = STFT_NPERSEG - STFT_NOVERLAP  # 128 samples

# -----------------------------------------------------------------------------
# Model hyper-parameters (from paper §2.5.2) (moved from CELL 6)
# -----------------------------------------------------------------------------
FEATURE_DIM     = 128         # D
N_ATTN_HEADS    = 4           # H
N_ATTN_LAYERS   = 2           # NA
CONV_KERNEL     = 3           # NC
CONV_GROUPS     = 3           # depth-wise separable groups
DROPout         = 0.5         # dropout rate
FC_HIDDEN       = 512         # HF

# -----------------------------------------------------------------------------
# Training hyper-parameters (moved from CELL 6)
# -----------------------------------------------------------------------------
BATCH_SIZE      = 128
NUM_EPOCHS      = 100
LR              = 1e-3
WEIGHT_DECAY    = 1e-2
WARMUP_EPOCHS   = 5
LABEL_SMOOTHING = 0.1
GRAD_CLIP       = 1.0
NUM_WORKERS     = 0           # Colab: keep 0 to avoid pickling issues
PIN_MEMORY      = True
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =============================================================================
# STFT time-frequency representation
# =============================================================================

def compute_stft_representation(
    signal: np.ndarray,          # (C, T)  float32, C=3 channels
    fs: int = SFREQ,
    nperseg: int = STFT_NPERSEG,
    noverlap: int = STFT_NOVERLAP,
) -> np.ndarray:
    """
    Compute log-power STFT for each channel.
    Returns array of shape (C, T_freq, F) where:
        T_freq = number of time frames
        F      = frequency bins (nperseg // 2 + 1)
    Values are log-power (dB-ish).
    """
    C = signal.shape[0]
    hop = nperseg - noverlap
    n_freq = nperseg // 2 + 1

    # Use scipy for STFT (vectorized across channels)
    from scipy.signal import stft

    specs = []
    for c in range(C):
        f, t, Z = stft(
            signal[c],
            fs=fs,
            window="hamming",
            nperseg=nperseg,
            noverlap=noverlap,
            boundary=None,
            padded=False,
        )
        # Z shape: (n_freq, n_time)
        power = np.abs(Z) ** 2
        log_power = np.log10(power + 1e-10)
        specs.append(log_power)

    # Stack → (C, n_freq, n_time), then transpose to (C, n_time, n_freq)
    out = np.stack(specs, axis=0)                 # (C, n_freq, n_time)
    out = np.transpose(out, (0, 2, 1))            # (C, n_time, n_freq)
    return out.astype(np.float32)


def compute_stft_batch(
    signals: np.ndarray,          # (B, C, T)
) -> np.ndarray:
    """Vectorized STFT over a batch. Returns (B, C, T_freq, F)."""
    B = signals.shape[0]
    outs = [compute_stft_representation(signals[i]) for i in range(B)]
    return np.stack(outs, axis=0)

# =============================================================================
# Compute the actual STFT output shape for our data (moved from CELL 4b)
# =============================================================================

dummy = np.zeros((N_CHANNELS, SAMPLES_PER_EPOCH), dtype=np.float32)
dummy_stft = compute_stft_representation(dummy)
T_FREQ = dummy_stft.shape[1]
F_BINS = dummy_stft.shape[2]

# =============================================================================
# PyTorch Dataset that loads shards on demand
# =============================================================================

class SleepEDFDataset(Dataset):
    """
    Loads all epochs belonging to a given set of subjects.
    Since shards are large (≈900 MB each), we load them once and keep
    the relevant slices in memory.

    We store the STFT representation (not raw signal) to save RAM.
    If RAM is tight, set `precompute_stft=False` and compute STFT on-the-fly.
    """

    def __init__(
        self,
        data_dir: Path,
        subject_set: set,
        *,
        precompute_stft: bool = True,
        normalize: bool = True,
        norm_stats: Optional[dict] = None,   # dict with 'mean' and 'std' per channel
    ):
        super().__init__()
        self.data_dir = Path(data_dir)
        self.subject_set = set(subject_set)
        self.precompute_stft = precompute_stft
        self.normalize = normalize
        self.norm_stats = norm_stats

        self._signals: list[np.ndarray] = []   # each: (n_i, C, 3000)
        self._stfts:   list[np.ndarray] = []   # each: (n_i, C, T_freq, F)
        self._labels:  list[np.ndarray] = []   # each: (n_i,)
        self._subjects: list[np.ndarray] = []  # each: (n_i,)

        self._load()

    def _load(self):
        for sp in sorted(self.data_dir.glob("shard_*.npz")):
            with np.load(sp, allow_pickle=False) as d:
                X = d["X"]                              # (n, 5, 3000)
                y = d["y"].astype(np.int64)              # (n,)
                subj = X[:, SUBJECT_IDX_IN_X, 0].astype(np.int16)

            mask = np.isin(subj, list(self.subject_set))
            if not mask.any():
                continue

            X_sel = X[mask][:, [0, 1, 2], :]             # keep 3 channels
            y_sel = y[mask]
            subj_sel = subj[mask]

            if self.precompute_stft:
                stft_sel = compute_stft_batch(X_sel)     # (n, C, T_freq, F)
                self._stfts.append(stft_sel)
            else:
                self._signals.append(X_sel)

            self._labels.append(y_sel)
            self._subjects.append(subj_sel)

        if self.precompute_stft:
            self._stfts = [np.concatenate(self._stfts, axis=0)]
        else:
            self._signals = [np.concatenate(self._signals, axis=0)]

        self._labels   = [np.concatenate(self._labels, axis=0)]
        self._subjects = [np.concatenate(self._subjects, axis=0)]

        if self.precompute_stft:
            self.n = self._stfts[0].shape[0]
        else:
            self.n = self._signals[0].shape[0]

        print(f"  loaded {self.n} epochs for subjects {sorted(self.subject_set)[:5]}...")

        # ---- Compute normalization stats if not provided --------------------
        if self.normalize and self.norm_stats is None:
            self._compute_norm_stats()

    def _compute_norm_stats(self):
        """Compute per-channel mean/std from the STFT features."""
        # Concatenate all STFT tensors: (n, C, T, F)
        arr = self._stfts[0]                       # (n, C, T, F)
        # mean/std over (n, T, F) → per channel
        mean = arr.mean(axis=(0, 2, 3))            # (C,)
        std  = arr.std(axis=(0, 2, 3)) + 1e-8       # (C,)
        self.norm_stats = {"mean": mean.astype(np.float32),
                           "std":  std.astype(np.float32)}

    def set_norm_stats(self, mean, std):
        self.norm_stats = {"mean": np.asarray(mean, dtype=np.float32),
                           "std":  np.asarray(std, dtype=np.float32)}

    def __len__(self):
        return self.n

    def __getitem__(self, idx: int):
        if self.precompute_stft:
            x = self._stfts[0][idx]                  # (C, T, F)
        else:
            sig = self._signals[0][idx]              # (C, 3000)
            x = compute_stft_representation(sig)      # (C, T, F)

        if self.normalize and self.norm_stats is not None:
            x = (x - self.norm_stats["mean"][:, None, None]) / \
                self.norm_stats["std"][:, None, None]

        y = self._labels[0][idx]
        subj = self._subjects[0][idx]
        return torch.from_numpy(x).float(), int(y), int(subj)


# =============================================================================
# MCAF-Net: Multi-Channel Attention Fusion Network
# =============================================================================
# Faithful implementation of:
#   Xu et al., "MCAF-Net: Multi-Channel Temporal Cross-Attention Network with
#   Dynamic Gating for Sleep Stage Classification", Sensors 2025, 25(14), 4251.
#
# Modules:
#   1. TemporalConv  — per-channel temporal feature extraction (two 1D convs)
#   2. MCAF          — dynamic gated multi-head cross-channel attention
#   3. Classification — two fully-connected layers
#
# Implementation details flagged as "ADAPTED" are not fully specified in the
# paper and required engineering decisions.
# =============================================================================


class TemporalConv(nn.Module):
    """
    Per-channel temporal feature extraction (paper §2.2).
    Input : (B, C, T, F)  →  transposed to (B, C, F, T)  →  1D conv over T
    Output: (B, C, T, D) where D = FEATURE_DIM
    """
    def __init__(self, in_dim: int, out_dim: int, kernel_size: int = 3,
                 dropout: float = 0.1):
        super().__init__()
        # First 1D conv: in_dim → 2*out_dim, kernel=3, padding=1
        self.conv1 = nn.Conv1d(
            in_channels=in_dim,
            out_channels=2 * out_dim,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            bias=True,
        )
        self.act1 = nn.GELU()
        # Second 1D conv: 2*out_dim → out_dim, kernel=1 (1×1)
        self.conv2 = nn.Conv1d(
            in_channels=2 * out_dim,
            out_channels=out_dim,
            kernel_size=1,
            bias=True,
        )
        self.act2 = nn.GELU()
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, C, T, F)  — batch, channels, time, freq
        returns: (B, C, T, D)
        """
        B, C, T, F = x.shape
        # Reshape to (B*C, F, T) for 1D conv over time
        x = x.reshape(B * C, T, F)          # (B*C, T, F)
        x = x.transpose(1, 2)                # (B*C, F, T)
        x = self.conv1(x)                    # (B*C, 2D, T)
        x = self.act1(x)
        x = self.dropout(x)
        x = self.conv2(x)                    # (B*C, D, T)
        x = self.act2(x)
        x = x.transpose(1, 2)                # (B*C, T, D)
        x = self.norm(x)                     # (B*C, T, D)
        x = x.reshape(B, C, T, -1)           # (B, C, T, D)
        return x


class DynamicGatedMCAF(nn.Module):
    """
    Multi-Channel Attention Fusion with dynamic gating (paper §2.3).

    Input : (B, C, L, D)  where L = time frames (T in paper)
    Output: (B, C, L, D)  fused cross-channel features

    Steps:
      1. Linear projection of input → Q, K, V  (shape: B, C, L, 3D)
      2. Split into H heads → (B, C, H, L, d_h)
      3. Cross-channel attention: for each pair of channels (i,j), compute
         attention from channel i's Q to channel j's K.
      4. Concatenate heads → (B, C, L, D)
      5. Linear projection WO
      6. Dynamic gate: G = sigmoid(mean(X) @ W_G + b);  Y = X + G * O'
    """
    def __init__(self, dim: int, n_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        self.dim = dim
        self.n_heads = n_heads
        self.d_head = dim // n_heads

        # QKV projection: D → 3D
        self.qkv = nn.Linear(dim, 3 * dim, bias=True)
        # Output projection: D → D
        self.proj = nn.Linear(dim, dim, bias=True)
        # Dynamic gating: D → 1
        self.gate = nn.Linear(dim, 1, bias=True)

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor):
        """
        x: (B, C, L, D)
        returns:
            out: (B, C, L, D)
            attn_weights: (B, C, H, L, L)  — for XAI
        """
        B, C, L, D = x.shape
        H, d_h = self.n_heads, self.d_head

        # ---- QKV projection ------------------------------------------------
        qkv = self.qkv(x)                       # (B, C, L, 3D)
        q, k, v = qkv.chunk(3, dim=-1)          # each (B, C, L, D)

        # ---- Reshape for multi-head ----------------------------------------
        def _to_heads(t):
            return t.reshape(B, C, L, H, d_h).permute(0, 1, 3, 2, 4)
            # → (B, C, H, L, d_h)

        q = _to_heads(q)                        # (B, C, H, L, d_h)
        k = _to_heads(k)
        v = _to_heads(v)

        # ---- Cross-channel attention ---------------------------------------
        # We want each channel to attend to ALL channels' features.
        # Approach: flatten channels into a single "sequence" dimension
        #   q_flat: (B, H, C*L, d_h)
        #   k_flat: (B, H, C*L, d_h)
        # This lets the attention mechanism model cross-channel dependencies.
        q_cross = q.permute(0, 2, 1, 3, 4).reshape(B, H, C * L, d_h)  # (B, H, C*L, d_h)
        k_cross = k.permute(0, 2, 1, 3, 4).reshape(B, H, C * L, d_h)
        v_cross = v.permute(0, 2, 1, 3, 4).reshape(B, H, C * L, d_h)

        # Attention scores: (B, H, C*L, C*L)
        attn_scores = torch.matmul(q_cross, k_cross.transpose(-2, -1)) / math.sqrt(d_h)
        attn_weights = F.softmax(attn_scores, dim=-1)            # (B, H, C*L, C*L)

        # Weighted sum: (B, H, C*L, d_h)
        attn_out = torch.matmul(attn_weights, v_cross)

        # Reshape back to (B, C, H, L, d_h) → (B, C, L, D)
        attn_out = attn_out.reshape(B, H, C, L, d_h)              # (B, H, C, L, d_h)
        attn_out = attn_out.permute(0, 2, 3, 1, 4).reshape(B, C, L, H * d_h)  # (B, C, L, D)

        # ---- Output projection ---------------------------------------------
        out = self.proj(attn_out)               # (B, C, L, D)
        out = self.dropout(out)

        # ---- Dynamic gating -------------------------------------------------
        # Gate input: mean over C and L → (B, D)
        x_mean = x.mean(dim=(1, 2))             # (B, D)
        gate = torch.sigmoid(self.gate(x_mean)) # (B, 1)
        gate = gate.reshape(B, 1, 1, 1)         # broadcastable

        # Residual connection
        out = x + gate * out                     # (B, C, L, D)
        out = self.norm(out)
        return out, attn_weights


class MCAFNet(nn.Module):
    """
    Full MCAF-Net architecture.

    Input : (B, C, T, F)  — log-power STFT of C channels
    Output: (B, N_CLASSES) logits
    """
    def __init__(
        self,n_channels: int = 3,
        n_classes: int = 5,
        input_freq_bins: int = 129,
        feature_dim: int = 128,
        n_heads: int = 4,
        n_attn_layers: int = 2,
        conv_kernel: int = 3,
        dropout: float = 0.5,
        fc_hidden: int = 512,
    ):
        super().__init__()
        self.n_channels = n_channels
        self.input_freq_bins = input_freq_bins

        # ---- TemporalConv per channel ---------------------------------------
        # One shared TemporalConv across channels (weight sharing),
        # OR one per channel? Paper says "each of the three channels was
        # processed by D dedicated temporal convolution modules"
        # → we use a single module but applied per-channel (weight sharing is
        #   a reasonable interpretation; alternatively use separate modules).
        self.temporal_conv = TemporalConv(
            in_dim=input_freq_bins,
            out_dim=feature_dim,
            kernel_size=conv_kernel,
            dropout=0.1,
        )

        # ---- MCAF layers ----------------------------------------------------
        self.mcaf_layers = nn.ModuleList([
            DynamicGatedMCAF(dim=feature_dim, n_heads=n_heads, dropout=0.1)
            for _ in range(n_attn_layers)
        ])

        # ---- Classification head -------------------------------------------
        # Flatten: (B, C, L, D) → (B, C*L*D) → but paper says (B, C*D)
        # Paper: "output Y ∈ R^{B×C×D} flattened to Y' ∈ R^{B×(C×D)}"
        # This implies they aggregate over L (time) first, e.g., global pooling.
        self.time_pool = nn.AdaptiveAvgPool2d((1, None))  # pool over L → (B, C, 1, D)
        self.classifier = nn.Sequential(
            nn.Linear(n_channels * feature_dim, fc_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(fc_hidden, n_classes),
        )

    def forward(self, x: torch.Tensor, return_attention: bool = False):
        """
        x: (B, C, T, F)  — log-power STFT
        returns: logits (B, n_classes), and optionally attention weights.
        """
        # ---- TemporalConv ---------------------------------------------------
        # x: (B, C, T, F) → (B, C, T, D)
        h = self.temporal_conv(x)

        # ---- MCAF layers ----------------------------------------------------
        attn_list = []
        for layer in self.mcaf_layers:
            h, attn = layer(h)                   # (B, C, T, D)
            attn_list.append(attn)

        # ---- Pool over time and classify -----------------------------------
        # h: (B, C, T, D)
        # Pool over T: mean
        h = h.mean(dim=2)                        # (B, C, D)
        h = h.reshape(h.size(0), -1)             # (B, C*D)
        logits = self.classifier(h)              # (B, n_classes)

        if return_attention:
            return logits, attn_list
        return logits


# =============================================================================
# Training and Evaluation functions
# =============================================================================

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for batch_idx, (X, y, _) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()

        # Gradient clipping
        if GRAD_CLIP > 0:
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        optimizer.step()
        total_loss += loss.item()

        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    kappa = cohen_kappa_score(all_labels, all_preds)

    return {"loss": avg_loss, "accuracy": accuracy, "f1_macro": f1_macro, "kappa": kappa}


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_idx, (X, y, _) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss = criterion(logits, y)
            total_loss += loss.item()

            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)
    kappa = cohen_kappa_score(all_labels, all_preds)

    return {"loss": avg_loss, "accuracy": accuracy, "f1_macro": f1_macro, "f1_per_class": f1_per_class, "kappa": kappa}, all_labels, all_preds


def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return float(epoch) / float(max(1, WARMUP_EPOCHS))
    return 1.0


# Load the generated splits
with open(SPLIT_PATH, 'r') as f:
    all_cv_splits = json.load(f)

all_fold_results = []

for fold_idx, fold_splits in enumerate(all_cv_splits):
    print(f"\n{'='*80}\nStarting Fold {fold_idx} / {N_FOLDS-1}\n{'='*80}")

    train_subjects = set(fold_splits['train_subjects'])
    val_subjects = set(fold_splits['validation_subjects'])
    test_subjects = set(fold_splits['test_subjects'])

    # Create a unique checkpoint directory for this fold
    fold_ckpt_dir = CKPT_BASE_DIR / f"fold_{fold_idx}"
    fold_ckpt_dir.mkdir(parents=True, exist_ok=True)

    # =========================================================================
    # Data Loading and Preprocessing for current fold
    # =========================================================================
    print("Building train dataset...")
    train_ds = SleepEDFDataset(DATA_DIR, train_subjects, precompute_stft=True)

    print("\nBuilding validation dataset...")
    val_ds = SleepEDFDataset(DATA_DIR, val_subjects, precompute_stft=True,
                             normalize=True, norm_stats=train_ds.norm_stats)

    print("\nBuilding test dataset...")
    test_ds = SleepEDFDataset(DATA_DIR, test_subjects, precompute_stft=True,
                              normalize=True, norm_stats=train_ds.norm_stats)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )

    print(f"\nTrain batches : {len(train_loader)}")
    print(f"Val batches   : {len(val_loader)}")
    print(f"Test batches  : {len(test_loader)}")

    # =========================================================================
    # Model, Optimizer, Scheduler, Loss for current fold
    # =========================================================================
    # Re-initialize model, optimizer, and scheduler for each fold
    # to ensure independent training

    # Class weights from the training set for this fold
    class_counts = np.bincount(train_ds._labels[0], minlength=N_CLASSES).astype(np.float64)
    class_weights = 1.0 / (class_counts + 1e-6)
    class_weights = class_weights / class_weights.sum() * N_CLASSES
    class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

    print(f"\nFold {fold_idx} Class counts  : {class_counts.astype(int).tolist()}")
    print(f"Fold {fold_idx} Class weights : {class_weights.round(4).tolist()}")

    model = MCAFNet(
        n_channels=N_CHANNELS,
        n_classes=N_CLASSES,
        input_freq_bins=F_BINS,
        feature_dim=FEATURE_DIM,
        n_heads=N_ATTN_HEADS,
        n_attn_layers=N_ATTN_LAYERS,
        conv_kernel=CONV_KERNEL,
        dropout=DROPout,
        fc_hidden=FC_HIDDEN,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights_t,
        label_smoothing=LABEL_SMOOTHING,
    )

    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print("\n✓ Model, optimizer, scheduler, criterion re-initialized for fold.")

    # =========================================================================
    # Training Loop for current fold
    # =========================================================================
    PATIENCE        = 100        # epochs without val-Macro-F1 improvement
    MIN_DELTA       = 1e-4       # minimum improvement to count
    early_stop_wait = 0
    best_val_f1      = -1.0
    best_model_state = None
    best_norm_stats = None

    fold_history = []
    t_start_fold = time.time()

    for epoch in range(NUM_EPOCHS):
        t0 = time.time()

        train_metrics = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_metrics, _, _ = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()

        lr_now = optimizer.param_groups[0]["lr"]
        dt = time.time() - t0

        fold_history.append({
            "epoch":          epoch,
            "lr":             lr_now,
            "train_loss":     train_metrics["loss"],
            "train_acc":      train_metrics["accuracy"],
            "train_f1_macro": train_metrics["f1_macro"],
            "train_kappa":    train_metrics["kappa"],
            "val_loss":       val_metrics["loss"],
            "val_acc":        val_metrics["accuracy"],
            "val_f1_macro":   val_metrics["f1_macro"],
            "val_kappa":      val_metrics["kappa"],
            "val_f1_W":       val_metrics["f1_per_class"][0],
            "val_f1_N1":      val_metrics["f1_per_class"][1],
            "val_f1_N2":      val_metrics["f1_per_class"][2],
            "val_f1_N3":      val_metrics["f1_per_class"][3],
            "val_f1_REM":     val_metrics["f1_per_class"][4],
        })

        print(
            f"[{epoch+1:02d}/{NUM_EPOCHS}] "
            f"lr={lr_now:.2e} | "
            f"train loss={train_metrics['loss']:.4f} f1m={train_metrics['f1_macro']:.4f} | "
            f"val loss={val_metrics['loss']:.4f} "
            f"f1m={val_metrics['f1_macro']:.4f} "
            f"κ={val_metrics['kappa']:.4f} | "
            f"{dt:.1f}s"
        )

        if val_metrics["f1_macro"] > best_val_f1 + MIN_DELTA:
            best_val_f1 = val_metrics["f1_macro"]
            early_stop_wait = 0
            best_model_state = deepcopy(model.state_dict())
            best_norm_stats = train_ds.norm_stats
            torch.save({
                "epoch":                epoch,
                "model_state_dict":     best_model_state,
                "optimizer_state_dict": optimizer.state_dict(), # ADAPTED: saving current optimizer state for restart
                "scheduler_state_dict": scheduler.state_dict(), # ADAPTED: saving current scheduler state
                "val_metrics":          val_metrics,
                "val_f1_macro":         best_val_f1,
                "config": {
                    "n_channels":      N_CHANNELS,
                    "n_classes":       N_CLASSES,
                    "input_freq_bins": F_BINS,
                    "feature_dim":     FEATURE_DIM,
                    "n_heads":         N_ATTN_HEADS,
                    "n_attn_layers":   N_ATTN_LAYERS,
                    "conv_kernel":     CONV_KERNEL,
                    "dropout":         DROPout,
                    "fc_hidden":       FC_HIDDEN,
                },
                "norm_stats": best_norm_stats,
                "fold_splits": fold_splits, # Save fold-specific splits
            }, fold_ckpt_dir / "best_model.pt")
            print(f"  ✓ New best val Macro F1 = {best_val_f1:.4f}  " \
                  f"(saved to {fold_ckpt_dir / 'best_model.pt'}) ")
        else:
            early_stop_wait += 1
            print(f"  no improvement ({early_stop_wait}/{PATIENCE})")

        if early_stop_wait >= PATIENCE:
            print(f"\n⏹ Early stopping triggered after {epoch+1} epochs " \
                  f"(no val Macro F1 improvement for {PATIENCE} epochs).")
            break

    total_time_fold = time.time() - t_start_fold
    print(f"\n✓ Fold {fold_idx} training finished in {total_time_fold/60:.1f} min")
    print(f"✓ Best validation Macro F1 for Fold {fold_idx}: {best_val_f1:.4f}")

    # Save history for the current fold
    fold_history_df = pd.DataFrame(fold_history)
    fold_history_df.to_csv(fold_ckpt_dir / "training_history.csv", index=False)
    print(f"✓ Training history for Fold {fold_idx} saved to {fold_ckpt_dir / 'training_history.csv'}")

    # =========================================================================
    # Test Evaluation for current fold
    # =========================================================================
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        model.eval() # Ensure model is in eval mode after loading state_dict

        test_metrics, test_labels, test_preds = evaluate(
            model, test_loader, criterion, DEVICE
        )

        print(f"\n{'='*30}\nTEST RESULTS FOR FOLD {fold_idx}\n{'='*30}")
        print(f"Macro F1 (primary)  : {test_metrics['f1_macro']:.4f}")
        print(f"Accuracy            : {test_metrics['accuracy']:.4f}")
        print(f"Cohen's κ           : {test_metrics['kappa']:.4f}")
        print(f"Loss                : {test_metrics['loss']:.4f}")

        fold_results = {
            "fold":                fold_idx,
            "best_epoch":          int(np.argmax(fold_history_df['val_f1_macro'])),
            "best_val_f1_macro":   float(best_val_f1),
            "test_f1_macro":       float(test_metrics["f1_macro"]),
            "test_accuracy":       float(test_metrics["accuracy"]),
            "test_kappa":          float(test_metrics["kappa"]),
            "test_loss":           float(test_metrics["loss"]),
            "test_f1_per_class":   test_metrics["f1_per_class"].tolist(),
            "confusion_matrix":    confusion_matrix(test_labels, test_preds, labels=list(range(N_CLASSES))).tolist(),
        }
        all_fold_results.append(fold_results)

        with open(fold_ckpt_dir / "test_results_fold.json", "w") as f:
            json.dump(fold_results, f, indent=2)
        print(f"✓ Test results for Fold {fold_idx} saved to {fold_ckpt_dir / 'test_results_fold.json'}")
    else:
        print(f"⚠ No best model checkpoint found for Fold {fold_idx}. Skipping test evaluation.")

    # Clear memory for the next fold
    del train_ds, val_ds, test_ds, train_loader, val_loader, test_loader
    del model, optimizer, scheduler, criterion
    torch.cuda.empty_cache()
    gc.collect()

# =========================================================================
# Aggregate and Report Final Cross-Validation Results
# =========================================================================
print(f"\n{'='*80}\nCross-Validation Complete! Aggregating Results...\n{'='*80}")

final_cv_df = pd.DataFrame(all_fold_results)
print(final_cv_df.to_string())

mean_f1_macro = final_cv_df['test_f1_macro'].mean()
std_f1_macro = final_cv_df['test_f1_macro'].std()

mean_acc = final_cv_df['test_accuracy'].mean()
std_acc = final_cv_df['test_accuracy'].std()

mean_kappa = final_cv_df['test_kappa'].mean()
std_kappa = final_cv_df['test_kappa'].std()

print(f"\nAverage Test Macro F1: {mean_f1_macro:.4f} ± {std_f1_macro:.4f}")
print(f"Average Test Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
print(f"Average Test Kappa: {mean_kappa:.4f} ± {std_kappa:.4f}")

# Save aggregated results
with open(CKPT_BASE_DIR / "cv_aggregated_results.json", "w") as f:
    json.dump({
        "all_fold_results": all_fold_results,
        "mean_test_f1_macro": mean_f1_macro,
        "std_test_f1_macro": std_f1_macro,
        "mean_test_accuracy": mean_acc,
        "std_test_accuracy": std_acc,
        "mean_test_kappa": mean_kappa,
        "std_test_kappa": std_kappa,
    }, f, indent=2)
print(f"✓ Aggregated CV results saved to {CKPT_BASE_DIR / 'cv_aggregated_results.json'}")



Starting Fold 0 / 4
Building train dataset...
  loaded 94178 epochs for subjects [1, 3, 4, 5, 8]...

Building validation dataset...
  loaded 22349 epochs for subjects [0, 6, 11, 15, 20]...

Building test dataset...
  loaded 31069 epochs for subjects [2, 7, 12, 17, 22]...

Train batches : 736
Val batches   : 175
Test batches  : 243

Fold 0 Class counts  : [11525, 14113, 44040, 8230, 16270]
Fold 0 Class weights : [1.1942, 0.9752, 0.3125, 1.6723, 0.8459]
Model parameters: 465,031

✓ Model, optimizer, scheduler, criterion re-initialized for fold.
[01/100] lr=2.00e-04 | train loss=1.7098 f1m=0.1745 | val loss=1.7156 f1m=0.1250 κ=-0.0229 | 15.0s
  ✓ New best val Macro F1 = 0.1250  (saved to /content/drive/MyDrive/Sleep/checkpoints_cv/fold_0/best_model.pt) 
[02/100] lr=4.00e-04 | train loss=0.9604 f1m=0.6938 | val loss=1.0511 f1m=0.6845 κ=0.5682 | 13.6s
  ✓ New best val Macro F1 = 0.6845  (saved to /content/drive/MyDrive/Sleep/checkpoints_cv/fold_0/best_model.pt) 
[03/100] lr=6.00e-04 | trai